# Finding the best calibration runs

Pick the best parameter sets by three criteria, **with all calibrated parameters returned**:

1. **Lowest SWE RMSE** &mdash; smallest `SWE_RMSE`.
2. **Lowest HNW relative bias** &mdash; `|HNW_Rel_BIAS|` closest to zero.
3. **Best combined** &mdash; mean of the two independent ranks (low SWE RMSE *and* HNW bias near zero).

Same single data source as the other ranking notebooks &mdash;
`hnw_validation/full_validation/all_summaries_validated_R.csv`.

Use the **`TOP_N` switch** below to list the best **3** or **5** runs for each criterion.

In [26]:
from pathlib import Path

# Project root - located through the .projectroot marker file, so this
# notebook runs from any checkout location.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / ".projectroot").exists())

import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")

PROJECT_ROOT = ROOT
CSV_PATH = PROJECT_ROOT / "hnw_validation" / "full_validation" / "all_summaries_validated_R.csv"

# calibrated parameters (returned in every best-run table)
PARAMS = ["rho_max", "rho_null", "eta_null", "k", "tau", "c_ov", "k_ov"]

df_raw = pd.read_csv(CSV_PATH)
print(f"loaded {len(df_raw)} rows from {CSV_PATH.relative_to(PROJECT_ROOT)}")

loaded 200 rows from hnw_validation/full_validation/all_summaries_validated_R.csv


## Switches

* **`TOP_N`** &mdash; how many best runs to list per criterion (set to **3** or **5**).
* The quality filters mirror the main ranking notebook so only well-converged runs with
  enough validation pairs are eligible. Set any to `None` to disable.

In [27]:
# =========================================================================== #
#  >>> SWITCH: how many best runs to show per criterion? (3 or 5) <<<          #
TOP_N = 40
# =========================================================================== #

# --- eligibility filters (same thresholds as all_calibration_results.ipynb) --
INCLUDE_DYN_RHO_MAX = False   # drop the dynamic-rho_max subset?
SCORE_MAX = 0.6               # keep only objective score < this (None to disable)
HNW_N_MIN = 30000            # min HNW validation pairs       (None to disable)
SWE_N_MIN = 400              # min SWE validation pairs       (None to disable)

df = df_raw.copy()
if not INCLUDE_DYN_RHO_MAX:
    df = df[df["subset"] != "dyn_rho_max"]
if SCORE_MAX is not None:
    df = df[df["best_value"] < SCORE_MAX]
if HNW_N_MIN is not None:
    df = df[df["HNW_N"] >= HNW_N_MIN]
if SWE_N_MIN is not None:
    df = df[df["SWE_N"] >= SWE_N_MIN]
df = df.dropna(subset=["SWE_RMSE", "HNW_Rel_BIAS"]).reset_index(drop=True)

print(f"TOP_N = {TOP_N}")
print(f"{len(df)} / {len(df_raw)} runs eligible after filtering")

TOP_N = 40
111 / 200 runs eligible after filtering


In [28]:
# --- build the three rankings ------------------------------------------------
df["abs_HNW_bias"] = df["HNW_Rel_BIAS"].abs()
df["rank_swe"] = df["SWE_RMSE"].rank(method="min")
df["rank_hnw"] = df["abs_HNW_bias"].rank(method="min")
df["rank_combined"] = (df["rank_swe"] + df["rank_hnw"]) / 2

ID_COLS = ["subset", "phase", "algorithm"]
METRIC_COLS = ["SWE_RMSE","HNW_RMSE", "SWE_Rel_BIAS", "HNW_Rel_BIAS", "abs_HNW_bias", "SWE_R2", "HNW_R2", "best_value"]
SHOW = ID_COLS + METRIC_COLS + PARAMS

def top(sort_col):
    out = df.sort_values(sort_col).head(TOP_N)[SHOW].reset_index(drop=True)
    out.index += 1
    return out.round(4)

best_swe      = top("SWE_RMSE")        # 1. lowest SWE RMSE
best_hnw      = top("abs_HNW_bias")    # 2. lowest |HNW relative bias|
best_combined = top("rank_combined")   # 3. best of both

## 1 &mdash; Lowest SWE RMSE

In [29]:
print(f"Best {TOP_N} runs by lowest SWE RMSE")
best_swe

Best 40 runs by lowest SWE RMSE


,subset,phase,algorithm,SWE_RMSE,HNW_RMSE,SWE_Rel_BIAS,HNW_Rel_BIAS,abs_HNW_bias,SWE_R2,HNW_R2,best_value,rho_max,rho_null,eta_null,k,tau,c_ov,k_ov
1,Win21,5E,Nelder-Mead,32.8356,2.9601,-0.0271,-0.1761,0.1761,0.9274,0.8289,0.4548,404.0513,81.6923,8.579863e+06,0.0301,0.0242,0.0005,0.3974
2,Win21,3B,DE,33.6649,3.1290,-0.0266,-0.1415,0.1415,0.9230,0.8059,0.1461,397.0777,79.3687,1.048380e+07,0.0864,0.0086,0.0024,0.5105
3,Win21,1A,DE,33.7982,3.2226,-0.0387,-0.1590,0.1590,0.9224,0.7941,0.2080,393.0527,74.7269,1.400441e+07,0.3648,0.0100,0.0023,0.1144
4,Win21,6C,Nelder-Mead,34.3598,2.9138,-0.0543,-0.1626,0.1626,0.9205,0.8342,0.2075,367.9536,85.8907,9.613063e+06,0.0304,0.0185,0.0006,0.4560
5,Rain_Gauge,2A,DE,34.7186,4.0748,0.0444,-0.2103,0.2103,0.9188,0.6758,0.1655,408.0289,53.3644,2.447481e+06,0.0267,0.0106,0.0001,0.7548
6,Rain_Gauge,2C,DE,35.7144,3.7134,0.0466,-0.1713,0.1713,0.9141,0.7307,0.1803,391.7115,60.8207,3.531213e+06,0.0255,0.0102,0.0008,0.8605
7,Rain_Gauge,2B,DE,35.8214,3.8473,0.0533,-0.1852,0.1852,0.9136,0.7110,0.1735,404.6641,58.1134,2.383090e+06,0.0282,0.0129,0.0007,0.8736
8,Rain_Gauge,3C,DE,35.8236,3.8466,0.0527,-0.1856,0.1856,0.9136,0.7111,0.1433,402.3849,58.4343,3.250824e+06,0.0253,0.0109,0.0003,0.6725
9,Rain_Gauge,6D,Nelder-Mead,36.0774,3.0882,0.0319,-0.0974,0.0974,0.9123,0.8138,0.1721,368.6752,80.3369,8.244306e+06,0.0217,0.0238,0.0006,0.4895
10,Win21,3B,Nelder-Mead,36.0870,2.9234,-0.0499,-0.1057,0.1057,0.9123,0.8331,0.5344,359.1366,99.5503,8.866513e+06,0.0411,0.0001,0.0005,0.5144


## 2 &mdash; Lowest HNW relative bias (closest to zero)

In [30]:
print(f"Best {TOP_N} runs by |HNW relative bias| closest to zero")
best_hnw

Best 40 runs by |HNW relative bias| closest to zero


,subset,phase,algorithm,SWE_RMSE,HNW_RMSE,SWE_Rel_BIAS,HNW_Rel_BIAS,abs_HNW_bias,SWE_R2,HNW_R2,best_value,rho_max,rho_null,eta_null,k,tau,c_ov,k_ov
1,below2000,2B,DE,53.3356,3.2598,0.1698,0.0012,0.0012,0.8084,0.7925,0.2017,420.7256,86.1742,3.197972e+06,0.0238,0.0282,0.0000,0.4961
2,all_alpsolut_staions,2A,DE,49.4548,2.9133,0.1531,-0.0025,0.0025,0.8353,0.8343,0.2139,428.4194,96.6336,7.994377e+06,0.0212,0.0261,0.0001,0.8322
3,Rain_Gauge,4A,Nelder-Mead,37.1335,2.9036,0.0676,-0.0025,0.0025,0.9071,0.8354,0.1517,391.8104,102.1516,8.665807e+06,0.0272,0.0231,0.0005,0.4028
4,Rain_Gauge,6B,Nelder-Mead,36.9947,2.9149,0.0640,0.0029,0.0029,0.9078,0.8341,0.1849,386.5197,102.7297,8.436644e+06,0.0273,0.0224,0.0006,0.4050
5,all_alpsolut_staions,3A,DE,48.6891,2.8618,0.1524,0.0040,0.0040,0.8403,0.8401,0.1724,430.2301,99.2178,9.664774e+06,0.0217,0.0138,0.0003,0.9838
6,Rain_Gauge,2C,Nelder-Mead,36.3646,2.9055,0.0541,-0.0047,0.0047,0.9109,0.8351,0.1941,380.0123,101.1664,8.232495e+06,0.0269,0.0227,0.0006,0.4117
7,all_alpsolut_staions,3B,DE,49.4727,2.8994,0.1548,0.0054,0.0054,0.8352,0.8358,0.1552,431.6740,96.4495,1.207480e+07,0.0212,0.0116,0.0007,0.8604
8,all_alpsolut_staions,1A,DE,49.7041,2.9337,0.1533,-0.0059,0.0059,0.8336,0.8319,0.2208,431.1895,92.5943,1.321145e+07,0.0202,0.0146,0.0009,0.9473
9,Rain_Gauge,2B,Nelder-Mead,36.1553,2.9084,0.0567,-0.0063,0.0063,0.9120,0.8348,0.1932,387.1172,102.0764,7.993883e+06,0.0286,0.0225,0.0006,0.4075
10,below2000,3A,DE,54.1418,3.0898,0.1760,0.0081,0.0081,0.8026,0.8136,0.1705,424.9554,88.4811,6.296492e+06,0.0215,0.0119,0.0004,0.7231


## 3 &mdash; Best combined (low SWE RMSE & HNW bias near zero)

In [31]:
print(f"Best {TOP_N} runs by combined rank (mean of SWE-RMSE rank & HNW-bias rank)")
display(best_combined)

Best 40 runs by combined rank (mean of SWE-RMSE rank & HNW-bias rank)


,subset,phase,algorithm,SWE_RMSE,HNW_RMSE,SWE_Rel_BIAS,HNW_Rel_BIAS,abs_HNW_bias,SWE_R2,HNW_R2,best_value,rho_max,rho_null,eta_null,k,tau,c_ov,k_ov
1,Rain_Gauge,2C,Nelder-Mead,36.3646,2.9055,0.0541,-0.0047,0.0047,0.9109,0.8351,0.1941,380.0123,101.1664,8.232495e+06,0.0269,0.0227,0.0006,0.4117
2,Rain_Gauge,2B,Nelder-Mead,36.1553,2.9084,0.0567,-0.0063,0.0063,0.9120,0.8348,0.1932,387.1172,102.0764,7.993883e+06,0.0286,0.0225,0.0006,0.4075
3,Rain_Gauge,6B,Nelder-Mead,36.9947,2.9149,0.0640,0.0029,0.0029,0.9078,0.8341,0.1849,386.5197,102.7297,8.436644e+06,0.0273,0.0224,0.0006,0.4050
4,Rain_Gauge,4A,Nelder-Mead,37.1335,2.9036,0.0676,-0.0025,0.0025,0.9071,0.8354,0.1517,391.8104,102.1516,8.665807e+06,0.0272,0.0231,0.0005,0.4028
5,Rain_Gauge,2A,Nelder-Mead,36.9913,2.8967,0.0630,-0.0161,0.0161,0.9078,0.8361,0.1892,388.3308,98.5538,9.187929e+06,0.0263,0.0226,0.0006,0.3212
6,Rain_Gauge,3A,Nelder-Mead,37.7811,2.9249,0.0718,0.0083,0.0083,0.9039,0.8329,0.1518,390.0108,103.0875,8.919157e+06,0.0268,0.0227,0.0006,0.3845
7,Rain_Gauge,3B,Nelder-Mead,37.9750,2.8850,0.0764,-0.0140,0.0140,0.9029,0.8375,0.1296,399.0678,99.5745,8.770342e+06,0.0264,0.0231,0.0005,0.4144
8,Rain_Gauge,5D,Nelder-Mead,37.1728,2.9364,0.0602,0.0173,0.0173,0.9069,0.8316,0.0978,378.6692,104.6730,8.766866e+06,0.0266,0.0226,0.0006,0.4027
9,Rain_Gauge,4B,Nelder-Mead,37.4620,2.8825,0.0704,-0.0176,0.0176,0.9055,0.8377,0.1494,395.3069,99.0019,9.023233e+06,0.0262,0.0233,0.0005,0.3926
10,Rain_Gauge,5A,Nelder-Mead,37.4759,2.9419,0.0636,0.0187,0.0187,0.9054,0.8310,0.1543,380.4475,104.5918,8.733047e+06,0.0266,0.0227,0.0006,0.3986


## Winners &mdash; single best run per criterion + parameters

## LaTeX tables &mdash; best SWE-RMSE run & best HNW-bias run

Two `booktabs` tables: the calibrated parameters plus the SWE / HNW metrics (RMSE, relative bias, R&sup2;) for the single best run under each criterion. The LaTeX source is printed below; copy it into the manuscript (`\usepackage{booktabs}` required).

In [32]:
# --- LaTeX rendering of parameters + metrics --------------------------------
def _sci(v, neg_ok=True):
    "Format v as $m\\times10^{e}$ math-mode LaTeX."
    s = f"{v:.3e}"                       # e.g. '8.580e+06' / '1.735e-05'
    mant, exp = s.split("e")
    exp = int(exp)                       # strips sign & leading zeros
    return rf"${mant}\times10^{{{exp}}}$"

# pretty LaTeX names + units + value formatters for each calibrated parameter
PAR_LATEX = {
    "rho_max":  (r"$\rho_{\max}$", r"kg\,m$^{-3}$",     lambda v: f"{v:.2f}"),
    "rho_null": (r"$\rho_{0}$",    r"kg\,m$^{-3}$",     lambda v: f"{v:.2f}"),
    "eta_null": (r"$\eta_{0}$",    r"Pa\,s",            _sci),
    "k":        (r"$k$",           r"day$^{-1}$",       lambda v: f"{v:.4f}"),
    "tau":      (r"$\tau$",        r"day",              lambda v: f"{v:.4f}"),
    "c_ov":     (r"$c_{ov}$",      r"m$^2$\,kg$^{-1}$", _sci),
    "k_ov":     (r"$k_{ov}$",      r"--",               lambda v: f"{v:.4f}"),
}

def latex_table(row, label, caption):
    L = [r"\begin{table}[ht]", r"  \centering",
         rf"  \caption{{{caption}}}", rf"  \label{{{label}}}",
         r"  \begin{tabular}{lrl}", r"    \toprule",
         r"    Quantity & Value & Unit \\", r"    \midrule",
         r"    \multicolumn{3}{l}{\textit{Calibrated parameters}} \\"]
    for p in PARAMS:
        name, unit, fmt = PAR_LATEX[p]
        L.append(rf"    {name} & {fmt(row[p])} & {unit} \\")
    L += [r"    \midrule",
          r"    \multicolumn{3}{l}{\textit{SWE performance}} \\",
          rf"    RMSE & {row['SWE_RMSE']:.2f} & kg\,m$^{{-2}}$ \\",
          rf"    Rel. bias & {row['SWE_Rel_BIAS']:+.4f} & -- \\",
          rf"    $R^2$ & {row['SWE_R2']:.3f} & -- \\",
          r"    \midrule",
          r"    \multicolumn{3}{l}{\textit{HNW performance}} \\",
          rf"    RMSE & {row['HNW_RMSE']:.2f} & mm \\",
          rf"    Rel. bias & {row['HNW_Rel_BIAS']:+.4f} & -- \\",
          rf"    $R^2$ & {row['HNW_R2']:.3f} & -- \\",
          r"    \bottomrule", r"  \end{tabular}", r"\end{table}"]
    return "\n".join(L)

# canonical subset display names (SP_all / SP_RG / SP_b2000)
SUBSET_DISPLAY = {"sp_all": "SP_all",
                  "Rain_Gauge": "SP_RG",
                  "below2000": "SP_b2000"}

# single best run for each criterion, straight from the (filtered) df
row_swe = df.sort_values("SWE_RMSE").iloc[0]
row_hnw = df.sort_values("abs_HNW_bias").iloc[0]

tab_swe = latex_table(
    row_swe, "tab:best_swe_rmse",
    rf"Best calibration run by SWE RMSE "
    rf"({SUBSET_DISPLAY.get(row_swe['subset'], row_swe['subset'])}, phase {row_swe['phase']}, {row_swe['algorithm']}): "
    rf"calibrated parameters and SWE/HNW validation metrics.")

tab_hnw = latex_table(
    row_hnw, "tab:best_hnw_bias",
    rf"Best calibration run by HNW relative bias "
    rf"({SUBSET_DISPLAY.get(row_hnw['subset'], row_hnw['subset'])}, phase {row_hnw['phase']}, {row_hnw['algorithm']}): "
    rf"calibrated parameters and SWE/HNW validation metrics.")

print(tab_swe)
print()
print(tab_hnw)


\begin{table}[ht]
  \centering
  \caption{Best calibration run by SWE RMSE (Win21, phase 5E, Nelder-Mead): calibrated parameters and SWE/HNW validation metrics.}
  \label{tab:best_swe_rmse}
  \begin{tabular}{lrl}
    \toprule
    Quantity & Value & Unit \\
    \midrule
    \multicolumn{3}{l}{\textit{Calibrated parameters}} \\
    $\rho_{\max}$ & 404.05 & kg\,m$^{-3}$ \\
    $\rho_{0}$ & 81.69 & kg\,m$^{-3}$ \\
    $\eta_{0}$ & $8.580\times10^{6}$ & Pa\,s \\
    $k$ & 0.0301 & day$^{-1}$ \\
    $\tau$ & 0.0242 & day \\
    $c_{ov}$ & $5.245\times10^{-4}$ & m$^2$\,kg$^{-1}$ \\
    $k_{ov}$ & 0.3974 & -- \\
    \midrule
    \multicolumn{3}{l}{\textit{SWE performance}} \\
    RMSE & 32.84 & kg\,m$^{-2}$ \\
    Rel. bias & -0.0271 & -- \\
    $R^2$ & 0.927 & -- \\
    \midrule
    \multicolumn{3}{l}{\textit{HNW performance}} \\
    RMSE & 2.96 & mm \\
    Rel. bias & -0.1761 & -- \\
    $R^2$ & 0.829 & -- \\
    \bottomrule
  \end{tabular}
\end{table}

\begin{table}[ht]
  \centering
  \capt

In [33]:
def describe(row, title):
    print(f"=== {title} ===")
    print(f"  subset={row['subset']}  phase={row['phase']}  algorithm={row['algorithm']}")
    print(f"  SWE_RMSE={row['SWE_RMSE']:.3f}   HNW_Rel_BIAS={row['HNW_Rel_BIAS']:+.4f}"
          f"   SWE_R2={row['SWE_R2']:.3f}   HNW_R2={row['HNW_R2']:.3f}   score={row['best_value']:.4f}")
    params = "  ".join(f"{p}={row[p]:g}" for p in PARAMS)
    print(f"  params: {params}\n")

describe(best_swe.iloc[0],      "Lowest SWE RMSE")
describe(best_hnw.iloc[0],      "Lowest |HNW relative bias|")
describe(best_combined.iloc[0], "Best combined")

=== Lowest SWE RMSE ===
  subset=Win21  phase=5E  algorithm=Nelder-Mead
  SWE_RMSE=32.836   HNW_Rel_BIAS=-0.1761   SWE_R2=0.927   HNW_R2=0.829   score=0.4548
  params: rho_max=404.051  rho_null=81.6923  eta_null=8.57986e+06  k=0.0301  tau=0.0242  c_ov=0.0005  k_ov=0.3974

=== Lowest |HNW relative bias| ===
  subset=below2000  phase=2B  algorithm=DE
  SWE_RMSE=53.336   HNW_Rel_BIAS=+0.0012   SWE_R2=0.808   HNW_R2=0.792   score=0.2017
  params: rho_max=420.726  rho_null=86.1742  eta_null=3.19797e+06  k=0.0238  tau=0.0282  c_ov=0  k_ov=0.4961

=== Best combined ===
  subset=Rain_Gauge  phase=2C  algorithm=Nelder-Mead
  SWE_RMSE=36.365   HNW_Rel_BIAS=-0.0047   SWE_R2=0.911   HNW_R2=0.835   score=0.1941
  params: rho_max=380.012  rho_null=101.166  eta_null=8.2325e+06  k=0.0269  tau=0.0227  c_ov=0.0006  k_ov=0.4117



## Scatter plots &mdash; top-10 runs vs default parameters & metric trade-off

The 10 best runs by **combined rank** (rank 1&rarr;10). One subplot per calibrated parameter: phase on the x-axis (ordered by rank), parameter value on the y-axis. The **&Delta;Snow default** is drawn as a dashed line in the &Delta;Snow colour. A final panel shows the **SWE RMSE vs HNW relative bias** trade-off. Markers encode the optimiser &mdash; **&times; = Nelder-Mead**, **&bull; = DE** &mdash; and point colour encodes rank.

In [34]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
sys.path.insert(0, str(PROJECT_ROOT))
from plot_style import apply_style, C, FIG, add_subplot_labels
apply_style()

PLOTS_BEST = PROJECT_ROOT / "calibration_ranking" / "plots_best_run"
PLOTS_BEST.mkdir(parents=True, exist_ok=True)

# default parameter values --- ΔSnow (winkler_snow_2021) & HS2SWE (magnusson_evaluating_2025)
DEFAULTS = {"rho_max": 401.26, "rho_null": 81.19, "eta_null": 8.52e6,
            "k": 0.03, "tau": 0.024, "c_ov": 5.1e-4, "k_ov": 0.379}
HS2SWE   = {"rho_max": 571.6, "rho_null": 113.7, "eta_null": 60.51e6, "k": 0.018}  # HS2SWE only defines these
# benchmark validation metrics (SWE RMSE, HNW rel. bias)
REF_DSNOW  = (33.9, -0.17)
REF_HS2SWE = (30.6,  0.03)

PAR_LABEL = {
    "rho_max":  r"$\rho_{\max}$ (kg m$^{-3}$)",
    "rho_null": r"$\rho_{0}$ (kg m$^{-3}$)",
    "eta_null": r"$\eta_{0}$ (Pa s)",
    "k":        r"$k$ (day$^{-1}$)",
    "tau":      r"$\tau$ (day)",
    "c_ov":     r"$c_{ov}$ (m$^2$ kg$^{-1}$)",
    "k_ov":     r"$k_{ov}$ (--)",
}
LOG_PARS   = {"eta_null", "c_ov"}
ALG_MARKER = {"Nelder-Mead": "X", "DE": "o"}   # X = NM, dot = DE
PAD_FRAC   = 0.15   # extra head/foot room below the min & above the max value

# --- 10 best runs by combined rank, ordered rank 1 -> 10 ---------------------
N_RANK = 50
rank10 = df.sort_values("rank_combined").head(N_RANK).reset_index(drop=True)
rank10["rank"] = np.arange(1, len(rank10) + 1)

fig, axes = plt.subplots(2, 4, figsize=(20, 9))

# --- one panel per calibrated parameter -------------------------------------
for idx, p in enumerate(PARAMS):
    ax = axes.flat[idx]
    ax.axhline(DEFAULTS[p], color=C.DSNOW, ls="--", lw=2.2, zorder=1)        # ΔSnow default
    if p in HS2SWE:
        ax.axhline(HS2SWE[p], color=C.HS2SWE, ls=":", lw=2.2, zorder=1)      # HS2SWE default
    for algo, mk in ALG_MARKER.items():
        s = rank10[rank10["algorithm"] == algo]
        if len(s):
            ax.scatter(s.index, s[p], marker=mk, s=80, color="k",
                       edgecolor="k", zorder=3)
    ax.set_xticks(range(len(rank10)))
    ax.set_xticklabels(rank10["phase"], rotation=45, ha="right", fontsize=8)
    ax.set_xlabel("phase (rank 1→10)")
    ax.set_ylabel(PAR_LABEL[p])
    ax.set_title(PAR_LABEL[p].split(" (")[0], fontweight="semibold")

    # --- y-limits with extra room below min & above max (incl. reference lines)
    vals = list(rank10[p].values) + [DEFAULTS[p]] + ([HS2SWE[p]] if p in HS2SWE else [])
    lo, hi = min(vals), max(vals)
    if p in LOG_PARS:
        ax.set_yscale("log")
        factor = (hi / lo) ** PAD_FRAC
        ax.set_ylim(lo / factor, hi * factor)
    else:
        pad = PAD_FRAC * (hi - lo) if hi > lo else abs(hi) * PAD_FRAC
        ax.set_ylim(lo - pad, hi + pad)

# --- final panel: SWE RMSE vs HNW relative bias trade-off -------------------
ax = axes.flat[7]
for algo, mk in ALG_MARKER.items():
    s = rank10[rank10["algorithm"] == algo]
    if len(s):
        ax.scatter(s["SWE_RMSE"], s["HNW_Rel_BIAS"], marker=mk, s=140, color="k",
                   edgecolor="k", linewidth=0.5, zorder=3)
ax.scatter(*REF_DSNOW,  marker="*", s=420, color=C.DSNOW,  edgecolor="k", linewidth=0.6, zorder=4)  # ΔSnow
ax.scatter(*REF_HS2SWE, marker="*", s=420, color=C.HS2SWE, edgecolor="k", linewidth=0.6, zorder=4)  # HS2SWE
ax.axhline(0, color="grey", lw=1, zorder=1)
ax.set_xlabel("SWE RMSE (kg m$^{-2}$)")
ax.set_ylabel("HNW rel. bias")
ax.set_title("SWE RMSE vs HNW rel. bias", fontweight="semibold")
ax.margins(x=0.12, y=0.15)   # extra room around the extreme points

# --- subplot labels a) .. h) -------------------------------------------------
add_subplot_labels(axes)

# --- title + legend (markers + benchmark references) -------------------------
legend_handles = [
    Line2D([0], [0], marker="X", color="w", markerfacecolor="k", markeredgecolor="k", markersize=13, label="Nelder-Mead"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="k", markeredgecolor="k", markersize=12, label="DE"),
    Line2D([0], [0], color=C.DSNOW,  ls="--", lw=2.2, label="ΔSnow default"),
    Line2D([0], [0], color=C.HS2SWE, ls=":",  lw=2.2, label="HS2SWE default"),
    Line2D([0], [0], marker="*", color="w", markerfacecolor=C.DSNOW,  markeredgecolor="k", markersize=16, label="ΔSnow benchmark"),
    Line2D([0], [0], marker="*", color="w", markerfacecolor=C.HS2SWE, markeredgecolor="k", markersize=16, label="HS2SWE benchmark"),
]
fig.legend(handles=legend_handles, loc="lower center", ncol=6, frameon=False,
           bbox_to_anchor=(0.5, 0.0))
fig.suptitle(f"Top {N_RANK} calibration runs by combined rank — parameters vs ΔSnow / HS2SWE defaults & metric trade-off",
             fontsize=15, fontweight="bold")
fig.tight_layout(rect=(0, 0.04, 1, 0.96))

fig.savefig(PLOTS_BEST / "top10_params_and_tradeoff.png", **FIG.SAVE)
print(f"saved -> {(PLOTS_BEST / 'top10_params_and_tradeoff.png').relative_to(PROJECT_ROOT)}")
plt.show()


saved -> calibration_ranking/plots_best_run/top10_params_and_tradeoff.png


In [35]:
rank10


,subset,dataset,phase,algorithm,w_SWE_NRMSE,w_RHO_NRMSE,w_SWE_NBIAS,w_RHO_NBIAS,w_SWE_KGE,w_RHO_KGE,...,HNW_RMSE,HNW_Bias,HNW_Rel_BIAS,HNW_R2,HNW_N,abs_HNW_bias,rank_swe,rank_hnw,rank_combined,rank
0,Rain_Gauge,SNOWPACK,2C,Nelder-Mead,0.30,0.70,0.00,0.00,0.0,0.0,...,2.905524,-0.009696,-0.004737,0.835147,35409,0.004737,13.0,6.0,9.5,1
1,Rain_Gauge,SNOWPACK,2B,Nelder-Mead,0.50,0.50,0.00,0.00,0.0,0.0,...,2.908388,-0.012954,-0.006330,0.834822,35409,0.006330,11.0,9.0,10.0,2
2,Rain_Gauge,SNOWPACK,6B,Nelder-Mead,0.50,0.00,0.00,0.00,0.0,0.5,...,2.914903,0.005939,0.002902,0.834081,35409,0.002902,18.0,4.0,11.0,3
3,Rain_Gauge,SNOWPACK,4A,Nelder-Mead,0.60,0.20,0.00,0.20,0.0,0.0,...,2.903565,-0.005077,-0.002481,0.835369,35409,0.002481,20.0,3.0,11.5,4
4,Rain_Gauge,SNOWPACK,2A,Nelder-Mead,0.70,0.30,0.00,0.00,0.0,0.0,...,2.896707,-0.032997,-0.016123,0.836146,35409,0.016123,16.0,17.0,16.5,5
5,Rain_Gauge,SNOWPACK,3A,Nelder-Mead,0.60,0.20,0.20,0.00,0.0,0.0,...,2.924862,0.017038,0.008325,0.832945,35409,0.008325,26.0,11.0,18.5,6
6,Rain_Gauge,SNOWPACK,3B,Nelder-Mead,0.70,0.00,0.30,0.00,0.0,0.0,...,2.885024,-0.028626,-0.013987,0.837465,35409,0.013987,27.0,13.0,20.0,7
7,Rain_Gauge,SNOWPACK,5D,Nelder-Mead,0.25,0.25,0.25,0.25,0.0,0.0,...,2.936399,0.035449,0.017321,0.831625,35409,0.017321,21.0,19.0,20.0,8
8,Rain_Gauge,SNOWPACK,4B,Nelder-Mead,0.70,0.10,0.00,0.20,0.0,0.0,...,2.882524,-0.035993,-0.017586,0.837747,35409,0.017586,22.0,20.0,21.0,9
9,Rain_Gauge,SNOWPACK,5A,Nelder-Mead,0.40,0.40,0.10,0.10,0.0,0.0,...,2.941865,0.038366,0.018746,0.830998,35409,0.018746,23.0,23.0,23.0,10


In [36]:
# --- normalise each metric by its mean, then build rankings -----------------
NORM_METRICS = ["SWE_RMSE", "abs_HNW_bias"]

df["abs_HNW_bias"] = df["HNW_Rel_BIAS"].abs()

metric_means = df[NORM_METRICS].mean()
for col in NORM_METRICS:
    df[f"{col}_norm"] = df[col] / metric_means[col]

df["rank_swe"] = df["SWE_RMSE_norm"].rank(method="min")
df["rank_hnw"] = df["abs_HNW_bias_norm"].rank(method="min")
df["rank_combined"] = ((df["SWE_RMSE_norm"] * 1.5) + (df["abs_HNW_bias_norm"] * 0.5)) / 2

ID_COLS = ["subset", "phase", "algorithm"]
METRIC_COLS = ["SWE_RMSE","HNW_RMSE", "SWE_Rel_BIAS", "HNW_Rel_BIAS", "abs_HNW_bias", "SWE_R2", "HNW_R2", "best_value"]
SHOW = ID_COLS + METRIC_COLS + PARAMS

def top(sort_col):
    out = df.sort_values(sort_col).head(100)[SHOW].reset_index(drop=True)
    out.index += 1
    return out.round(4)

best_swe      = top("SWE_RMSE")        # 1. lowest SWE RMSE


In [39]:
best_swe_filtered = df[df["HNW_Rel_BIAS"].abs() < 0.13].sort_values("SWE_RMSE").head(TOP_N)[SHOW].reset_index(drop=True)
best_swe_filtered.index += 1
best_swe_filtered = best_swe_filtered.round(4)

In [40]:
best_swe_filtered

,subset,phase,algorithm,SWE_RMSE,HNW_RMSE,SWE_Rel_BIAS,HNW_Rel_BIAS,abs_HNW_bias,SWE_R2,HNW_R2,best_value,rho_max,rho_null,eta_null,k,tau,c_ov,k_ov
1,Rain_Gauge,6D,Nelder-Mead,36.0774,3.0882,0.0319,-0.0974,0.0974,0.9123,0.8138,0.1721,368.6752,80.3369,8.244306e+06,0.0217,0.0238,0.0006,0.4895
2,Win21,3B,Nelder-Mead,36.0870,2.9234,-0.0499,-0.1057,0.1057,0.9123,0.8331,0.5344,359.1366,99.5503,8.866513e+06,0.0411,0.0001,0.0005,0.5144
3,Rain_Gauge,2B,Nelder-Mead,36.1553,2.9084,0.0567,-0.0063,0.0063,0.9120,0.8348,0.1932,387.1172,102.0764,7.993883e+06,0.0286,0.0225,0.0006,0.4075
4,Rain_Gauge,2C,Nelder-Mead,36.3646,2.9055,0.0541,-0.0047,0.0047,0.9109,0.8351,0.1941,380.0123,101.1664,8.232495e+06,0.0269,0.0227,0.0006,0.4117
5,Rain_Gauge,2A,Nelder-Mead,36.9913,2.8967,0.0630,-0.0161,0.0161,0.9078,0.8361,0.1892,388.3308,98.5538,9.187929e+06,0.0263,0.0226,0.0006,0.3212
6,Rain_Gauge,4C,Nelder-Mead,36.9928,2.9583,0.0550,0.0255,0.0255,0.9078,0.8291,0.1541,374.0697,106.1973,8.733750e+06,0.0269,0.0222,0.0006,0.4000
7,Rain_Gauge,6B,Nelder-Mead,36.9947,2.9149,0.0640,0.0029,0.0029,0.9078,0.8341,0.1849,386.5197,102.7297,8.436644e+06,0.0273,0.0224,0.0006,0.4050
8,Rain_Gauge,1B,Nelder-Mead,37.0794,3.0689,0.0525,-0.0911,0.0911,0.9074,0.8161,0.1929,381.2134,80.9173,8.834301e+06,0.0217,0.0223,0.0006,0.4181
9,Rain_Gauge,4A,Nelder-Mead,37.1335,2.9036,0.0676,-0.0025,0.0025,0.9071,0.8354,0.1517,391.8104,102.1516,8.665807e+06,0.0272,0.0231,0.0005,0.4028
10,Rain_Gauge,5D,Nelder-Mead,37.1728,2.9364,0.0602,0.0173,0.0173,0.9069,0.8316,0.0978,378.6692,104.6730,8.766866e+06,0.0266,0.0226,0.0006,0.4027
